# Cross-modality pathway enrichment comparison — CosMx vs scRNA-seq

**Goal:** Identify pathways concordantly enriched or depleted across CosMx spatial
transcriptomics and Salcher scRNA-seq atlas malignant cells. Concordance across
two independent modalities strengthens the biological interpretation. Both t-stat
ranked GSEA and per-cell scoring approaches are compared.

**Input:** GSEA results from `cancer_cell_pathway_analysis*.ipynb` and
`scrna_pathway_analysis*.ipynb` — 8 CSV files covering Hallmark and CosMx pathway
gene sets across both modalities and both scoring approaches.

**Output:** Lollipop, bubble, and heatmap figures for concordant pathway enrichment.
Primary figure panel: Hallmark t-test comparison lollipop plot.

## Analysis approach notes

Two GSEA scoring approaches are compared in this notebook:

**T-stat ranked GSEA** (`*_t_test.csv`) — genes are ranked by t-test scores
from differential expression analysis between MHC II positive and negative
epithelial cells. GSEA is then run on this pre-ranked gene list. This is the
primary approach used for paper figures as it directly uses the differential
expression signal and produces more interpretable NES scores.

**Per-cell scoring** (`*_single_cell.csv`) — `dc.mt.gsea` scores each cell
individually against each pathway, then Mann-Whitney U tests aggregate to
group level. This approach produces log2FC values that are very small in
magnitude (±0.1) due to averaging across hundreds of thousands of cells,
making biological interpretation harder. Kept for completeness and comparison.

**Primary figure panels:** Hallmark t-stat lollipop (`fig3_hallmark_tstat_lollipop.pdf`)
and CosMx pathways t-stat lollipop (`figS3_cosmx_pathways_tstat_lollipop.pdf`).

**Gene set collections:**
- Hallmark — 50 curated MSigDB gene sets representing well-defined biological states
- CosMx pathways — NanoString panel-specific gene sets for the CosMx 1000-plex panel

## Setup

In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

table_dir = repo_root / cfg['outputs']['tables']
fig_dir   = repo_root / cfg['outputs']['figures']

table_out = table_dir / 'figure3'
fig_out   = fig_dir   / 'figure3'
supp_out  = fig_dir   / 'figure3-supp'

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)

## Load GSEA results

All 8 GSEA result files are loaded — two modalities (CosMx, scRNA), two gene
set collections (Hallmark, CosMx pathways), and two scoring approaches
(t-stat ranked, per-cell scoring).

In [ ]:
# --- T-stat ranked GSEA results ---
cosmx_hallmark_tstat  = pd.read_csv(table_out / 'cosmx_gsea_hallmark_t_test.csv', index_col=0)
scrna_hallmark_tstat  = pd.read_csv(table_out / 'scrna_gsea_hallmark_t_test.csv', index_col=0)
cosmx_cosmx_tstat     = pd.read_csv(table_out / 'cosmx_gsea_cosmx_pathways_t_test.csv', index_col=0)
scrna_cosmx_tstat     = pd.read_csv(table_out / 'scrna_gsea_cosmx_pathways_t_test.csv', index_col=0)

# --- Per-cell scoring GSEA results ---
cosmx_hallmark_sc = pd.read_csv(table_out / 'cosmx_gsea_hallmark_single_cell.csv', index_col=0)
scrna_hallmark_sc = pd.read_csv(table_out / 'scrna_gsea_hallmark_single_cell.csv', index_col=0)
cosmx_cosmx_sc    = pd.read_csv(table_out / 'cosmx_gsea_cosmx_pathways_single_cell.csv', index_col=0)
scrna_cosmx_sc    = pd.read_csv(table_out / 'scrna_gsea_cosmx_pathways_single_cell.csv', index_col=0)

# standardize index name across all files
for df in [cosmx_hallmark_tstat, scrna_hallmark_tstat, cosmx_cosmx_tstat, scrna_cosmx_tstat,
           cosmx_hallmark_sc, scrna_hallmark_sc, cosmx_cosmx_sc, scrna_cosmx_sc]:
    df.index.name = 'source'

print("Loaded all 8 GSEA result files")
print(f"Hallmark pathways: {len(cosmx_hallmark_tstat)}")
print(f"CosMx pathways: {len(cosmx_cosmx_tstat)}")

## Helper functions

Three visualization functions are used throughout:

- `make_bubble_df` — merges CosMx and scRNA results and computes concordance metrics
- `plot_bubble` — NES CosMx vs NES scRNA scatter, size=significance, color=direction
- `plot_lollipop` — concordantly enriched/depleted pathways only, ranked by mean NES
- `plot_heatmap` — all pathways side by side across both datasets

In [ ]:
def make_bubble_df(cosmx_df, scrna_df, score_col='NES'):
    """
    Merge CosMx and scRNA GSEA results and compute concordance metrics.
    Works with both NES (t-stat) and log2fc (per-cell) columns.
    """
    cosmx_use = cosmx_df[[score_col, 'padj']].copy()
    scrna_use  = scrna_df[[score_col, 'padj']].copy()

    bubble = cosmx_use.merge(scrna_use, left_index=True, right_index=True,
                             suffixes=('_CosMx', '_sc'))
    bubble.index.name = 'source'
    bubble = bubble.reset_index()

    bubble['mean_score']   = bubble[[f'{score_col}_CosMx', f'{score_col}_sc']].mean(axis=1)
    bubble['mean_padj']    = bubble[['padj_CosMx', 'padj_sc']].mean(axis=1)
    bubble['neglog10FDR']  = -np.log10(bubble['mean_padj'].replace(0, np.nextafter(0, 1)))
    bubble['color']        = np.where(bubble['mean_score'] >= 0,
                                      cmap_high_low[0], cmap_high_low[1])
    bubble['alpha']        = np.clip(
        np.abs(bubble['mean_score']) / np.abs(bubble['mean_score']).max(), 0.2, 1
    )
    bubble['sizes'] = 30 + 270 * (
        (bubble['neglog10FDR'] - bubble['neglog10FDR'].min()) /
        (bubble['neglog10FDR'].max() - bubble['neglog10FDR'].min())
    )
    # cap neglog10FDR to avoid floored p-values dominating sizes
    bubble['neglog10FDR'] = -np.log10(bubble['mean_padj'].replace(0, np.nextafter(0, 1)))
    bubble['neglog10FDR'] = bubble['neglog10FDR'].clip(upper=10)  # cap at 10
    
# quadrant assignment
    def quadrant(row):
        x = row[f'{score_col}_CosMx']
        y = row[f'{score_col}_sc']
        if x >= 0 and y >= 0: return 'Q1 (+,+)'
        if x <  0 and y >= 0: return 'Q2 (-,+)'
        if x <  0 and y <  0: return 'Q3 (-,-)'
        return 'Q4 (+,-)'

    bubble['quadrant'] = bubble.apply(quadrant, axis=1)
    return bubble, score_col


def plot_bubble(bubble_df, score_col='NES', top_n_labels=5,
                label_discordant=False, sig_thresh=0.05,
                title=None, figsize=(9, 8)):
    """Bubble scatter with cleaner labels — only labels concordant quadrants.
    Set label_discordant=True to also label significant discordant pathways."""
    fig, ax = plt.subplots(figsize=figsize)

    for _, row in bubble_df.iterrows():
        ax.scatter(row[f'{score_col}_CosMx'], row[f'{score_col}_sc'],
                   s=row['sizes'], color=row['color'],
                   alpha=row['alpha'], edgecolor='none')

    # label top concordant pathways
    texts = []
    q1 = bubble_df[bubble_df['quadrant'] == 'Q1 (+,+)'].nlargest(top_n_labels, 'mean_score')
    q3 = bubble_df[bubble_df['quadrant'] == 'Q3 (-,-)'].nsmallest(top_n_labels, 'mean_score')
    to_label = pd.concat([q1, q3])

    # optionally label significant discordant pathways
    if label_discordant:
        score_thresh = 0.1 if score_col == 'log2fc' else 1.0
        q2_q4 = bubble_df[
            bubble_df['quadrant'].isin(['Q2 (-,+)', 'Q4 (+,-)']) &
            (
                (bubble_df['padj_CosMx'] < sig_thresh) |
                (bubble_df['padj_sc'] < sig_thresh)
            ) &
            (
                (bubble_df[f'{score_col}_CosMx'].abs() > score_thresh) |
                (bubble_df[f'{score_col}_sc'].abs() > score_thresh)
            )
        ]
        to_label = pd.concat([to_label, q2_q4])

    for _, row in to_label.iterrows():
        label = (row['source']
                 .replace('CosMx: ', '')
                 .replace('_', ' ')
                 .title())
        color = 'black' if row['quadrant'] in ['Q1 (+,+)', 'Q3 (-,-)'] else 'darkred'
        texts.append(ax.text(
            row[f'{score_col}_CosMx'], row[f'{score_col}_sc'],
            label, fontsize=9, color=color,
        ))

    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        adjust_text(
            texts,
            arrowprops=dict(arrowstyle='-', color='gray', lw=0.5,
                            connectionstyle='arc3,rad=0'),
            expand_points=(2, 2),
            expand_text=(2, 2),
            ax=ax,
        )

    ax.axhline(0, ls='--', c='gray')
    ax.axvline(0, ls='--', c='gray')
    lim = max(bubble_df[[f'{score_col}_CosMx', f'{score_col}_sc']].abs().max()) * 1.15
    lim = max(lim, 1.5 if score_col == 'NES' else 0.2)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.plot([-lim, lim], [-lim, lim], ls='--', c='black', lw=1)

    xlabel = 'NES (CosMx)' if score_col == 'NES' else 'Log2FC (CosMx)'
    ylabel = 'NES (scRNA)' if score_col == 'NES' else 'Log2FC (scRNA)'
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_title(title or 'Pathway Concordance: CosMx vs scRNA', fontsize=13)

    legend_elements = [
        Patch(facecolor=cmap_high_low[0], label='MHC-II High', edgecolor='none'),
        Patch(facecolor=cmap_high_low[1], label='MHC-II Low', edgecolor='none'),
    ]
    ax.legend(handles=legend_elements, title='Direction',
              bbox_to_anchor=(1.05, 1), loc='upper left')
    sns.despine()
    plt.tight_layout()
    return fig

def print_bubble_table(bubble_df, score_col='NES'):
    """Print a readable table of all pathways with their quadrant and scores."""
    cols = ['source', f'{score_col}_CosMx', f'{score_col}_sc',
            'mean_score', 'mean_padj', 'quadrant']
    table = (
        bubble_df[cols]
        .copy()
        .sort_values('quadrant')
        .round(3)
    )
    table['source'] = table['source'].str.replace('CosMx: ', '').str.replace('_', ' ').str.title()
    table.columns = ['Pathway', f'{score_col} CosMx', f'{score_col} scRNA',
                     f'Mean {score_col}', 'Mean padj', 'Quadrant']
    print(table.to_string(index=False))
    return table


def plot_lollipop(bubble_df, score_col='NES', title=None, figsize=None):
    """Lollipop of concordantly enriched/depleted pathways (Q1 and Q3 only).
    Shows all concordant pathways — concordance across modalities is the criterion,
    not significance threshold."""

    # filter to concordant quadrants only — no significance filter
    subset = bubble_df[bubble_df['quadrant'].isin(['Q1 (+,+)', 'Q3 (-,-)'])].copy()
    subset = subset.sort_values('mean_score', ascending=False).reset_index(drop=True)

    # normalize bubble sizes — wider range for visibility
    neglog_scaled = (subset['neglog10FDR'] - subset['neglog10FDR'].min()) / (
        (subset['neglog10FDR'].max() - subset['neglog10FDR'].min()) + 1e-9
    )
    subset['bubble_size'] = 80 + neglog_scaled * 500

    # clean pathway names
    subset['label'] = (
        subset['source']
        .str.replace('CosMx: ', '', regex=False)
        .str.replace('_', ' ')
        .str.title()
    )

    if figsize is None:
        figsize = (9, max(6, len(subset) * 0.35))
    
    fig, ax = plt.subplots(figsize=figsize)

    for i, row in enumerate(subset.itertuples()):
        ax.plot([0, row.mean_score], [i, i], color='gray', lw=1, zorder=1)

    ax.scatter(subset['mean_score'], range(len(subset)),
               s=subset['bubble_size'],
               c=subset['color'],
               alpha=0.85, edgecolor='none', zorder=2)

    x_max = subset['mean_score'].abs().max() * 1.2
    ax.set_xlim(-x_max, x_max)
    ax.set_yticks(range(len(subset)))
    ax.set_yticklabels(subset['label'], fontsize=14)
    ax.axvline(0, ls='--', color='gray', lw=1)
    xlabel = 'Mean NES' if score_col == 'NES' else 'Mean Log2FC'
    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_title(
        title or 'Concordantly Enriched and Depleted Pathways\n'
        'Color = Direction, Bubble Size = $-log_{10}$(FDR)',
        fontsize=13
    )
    ax.invert_yaxis()
    sns.despine()
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    
    legend_elements = [
    Patch(facecolor=cmap_high_low[0], label='MHC-II High', edgecolor='none'),
    Patch(facecolor=cmap_high_low[1], label='MHC-II Low', edgecolor='none'),
]

    # add representative bubble sizes with actual FDR values
    for fdr_label, fdr_val in [('0.5', 0.5), ('0.05', 0.05), ('< 0.001', 0.0001)]:
        neglog = -np.log10(float(fdr_val))
        neglog_sqrt_val = np.sqrt(neglog)
        size = 80 + 500 * (neglog_sqrt_val - np.sqrt(subset['neglog10FDR'].min())) / (
            np.sqrt(subset['neglog10FDR'].max()) - np.sqrt(subset['neglog10FDR'].min()) + 1e-9
        )
        size = np.clip(size, 10, 580)
        legend_elements.append(
            Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='gray', markeredgecolor='none',
                   markersize=np.sqrt(size) * 0.6,
                   label=f'FDR = {fdr_label}')
        )
    
    ax.legend(handles=legend_elements,
              title='Direction / $-log_{10}$(FDR)',
              bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    plt.tight_layout()
    return fig, subset

def plot_heatmap(cosmx_df, scrna_df, score_col='NES', title=None, figsize=(5, 10)):
    """Heatmap of all pathways, CosMx and scRNA side by side."""
    pivot = pd.DataFrame({
        'CosMx': cosmx_df[score_col],
        'scRNA':  scrna_df[score_col],
    })
    pivot['mean'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('mean', ascending=False).drop(columns='mean')

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(pivot, cmap='RdBu_r', center=0,
                cbar_kws={'label': 'NES' if score_col == 'NES' else 'Log2FC'},
                ax=ax)
    ax.set_title(title or 'Pathway Enrichment Across Datasets')
    ax.set_ylabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticks([i + 0.5 for i in range(len(pivot))])
    ax.set_yticklabels(
        [s.replace('CosMx: ', '').replace('_', ' ').title() for s in pivot.index],
        rotation=0, fontsize=12
    )
    plt.tight_layout()
    return fig

## Hallmark — t-stat comparison

Primary comparison: t-stat ranked GSEA on Hallmark gene sets. This is the most
principled approach and is used for the primary figure panel. Pathways in Q1
(enriched in both) and Q3 (depleted in both) are the concordantly regulated
pathways that appear across both spatial and single-cell modalities.

In [ ]:
bubble_hallmark_tstat, score_col = make_bubble_df(
    cosmx_hallmark_tstat, scrna_hallmark_tstat, score_col='NES'
)

print(f"Q1 (+,+) concordantly enriched: {(bubble_hallmark_tstat['quadrant']=='Q1 (+,+)').sum()}")
print(f"Q3 (-,-) concordantly depleted: {(bubble_hallmark_tstat['quadrant']=='Q3 (-,-)').sum()}")

# bubble plot + table
fig = plot_bubble(
    bubble_hallmark_tstat, label_discordant=False,
    title='Hallmark Pathway Concordance (t-stat)\nCosMx vs scRNA epithelial cells',
)
fig.savefig(supp_out / 'figS3_hallmark_tstat_bubble.pdf', bbox_inches='tight')
plt.show()

# print table for reference
bubble_table = print_bubble_table(bubble_hallmark_tstat, score_col='NES')
bubble_table.to_csv(table_out / 'hallmark_tstat_bubble_table.csv', index=False)

# lollipop
fig, concordant_subset = plot_lollipop(
    bubble_hallmark_tstat,
    title='Concordant Hallmark Pathways (t-stat)\nColor = Direction, Size = $-log_{10}$(FDR)', figsize= (14,10)
)
fig.savefig(fig_out / 'fig3_hallmark_tstat_lollipop.pdf', bbox_inches='tight')
plt.show()

# save concordant pathway table
concordant_subset[['source', 'NES_CosMx', 'NES_sc', 'mean_score', 'mean_padj', 'quadrant']].to_csv(
    table_out / 'concordant_hallmark_pathways_tstat.csv', index=False
)
print(f"\nConcordant significant pathways: {len(concordant_subset)}")

fig = plot_heatmap(
    cosmx_hallmark_tstat, scrna_hallmark_tstat,
    title='Hallamrk Pathway Enrichment: CosMx vs scRNA (t-stat)',
    figsize=(7, 15),
)
fig.savefig(supp_out / 'figS3_hallmark_pathways_tstat_heatmap.pdf', bbox_inches='tight')
plt.show()

## Hallmark — per-cell scoring comparison

Secondary comparison using per-cell Mann-Whitney scores. Uses log2fc instead
of NES. Shown for comparison with the t-stat approach.

In [ ]:
bubble_hallmark_sc, score_col_sc = make_bubble_df(
    cosmx_hallmark_sc, scrna_hallmark_sc, score_col='log2fc'
)

fig = plot_bubble(
    bubble_hallmark_sc, score_col='log2fc',
    title='Hallmark Pathway Concordance (per-cell)\nCosMx vs scRNA epithelial cells',
)
bubble_table = print_bubble_table(bubble_hallmark_sc, score_col='log2fc')
fig.savefig(supp_out / 'figS3_hallmark_sc_bubble.pdf', bbox_inches='tight')
plt.show()

fig, concordant_subset  = plot_lollipop(
    bubble_hallmark_sc, score_col='log2fc',
    title='Concordant Hallmark Pathways (per-cell)',figsize= (14,10)
)
fig.savefig(supp_out / 'figS3_hallmark_sc_lollipop.pdf', bbox_inches='tight')
plt.show()

## CosMx pathways — t-stat comparison

Same comparison using NanoString CosMx pathway annotations instead of Hallmark.
These are panel-specific gene sets curated by NanoString for the CosMx 1000-plex
panel, providing a targeted view of oncogenic signaling pathways.

In [ ]:
bubble_cosmx_tstat, _ = make_bubble_df(
    cosmx_cosmx_tstat, scrna_cosmx_tstat, score_col='NES'
)

fig = plot_bubble(
    bubble_cosmx_tstat,
    title='CosMx Pathway Concordance (t-stat)\nCosMx vs scRNA epithelial cells',
)
fig.savefig(supp_out / 'figS3_cosmx_pathways_tstat_bubble.pdf', bbox_inches='tight')
plt.show()

bubble_table = print_bubble_table(bubble_cosmx_tstat, score_col='NES')

fig, concordant_subset  = plot_lollipop(
    bubble_cosmx_tstat,
    title='Concordant CosMx Pathways (t-stat)',figsize = (14,10)
)
fig.savefig(supp_out / 'figS3_cosmx_pathways_tstat_lollipop.pdf', bbox_inches='tight')
plt.show()

fig = plot_heatmap(
    cosmx_cosmx_tstat, scrna_cosmx_tstat,
    title='CosMx Pathway Enrichment: CosMx vs scRNA (t-stat)',
    figsize=(5, 15),
)
fig.savefig(supp_out / 'figS3_cosmx_pathways_tstat_heatmap.pdf', bbox_inches='tight')
plt.show()

## CosMx pathways — per-cell scoring comparison

In [ ]:
bubble_cosmx_sc, _ = make_bubble_df(
    cosmx_cosmx_sc, scrna_cosmx_sc, score_col='log2fc'
)

fig = plot_bubble(
    bubble_cosmx_sc, score_col='log2fc',
    title='CosMx Pathway Concordance (per-cell)',
)
fig.savefig(supp_out / 'figS3_cosmx_pathways_sc_bubble.pdf', bbox_inches='tight')
plt.show()

fig, concordant_subset  = plot_lollipop(
    bubble_cosmx_sc, score_col='log2fc',
    title='Concordant CosMx Pathways (per-cell)',figsize = (14,10)
)
fig.savefig(supp_out / 'figS3_cosmx_pathways_sc_lollipop.pdf', bbox_inches='tight')
plt.show()

## Summary table

Concordantly enriched and depleted pathways (Q1 and Q3) from the primary
Hallmark t-stat comparison, saved as a table for the supplement.

In [ ]:
concordant = bubble_hallmark_tstat[
    bubble_hallmark_tstat['quadrant'].isin(['Q1 (+,+)', 'Q3 (-,-)'])
][['source', 'NES_CosMx', 'NES_sc', 'mean_score', 'mean_padj', 'quadrant']].copy()

concordant.columns = ['Pathway', 'NES_CosMx', 'NES_scRNA', 'Mean_NES', 'Mean_padj', 'Quadrant']
concordant = concordant.sort_values('Mean_NES', ascending=False)
concordant.to_csv(table_out / 'concordant_hallmark_pathways_tstat.csv', index=False)

print(f"Concordant pathways: {len(concordant)}")
print(concordant.to_string(index=False))